### Import packages

In [1]:
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path
import fitz
import json

### Import 2026 budget report pdf

In [2]:
def process_pdf_by_paragraph(pdf_filename):
    """Locates a PDF inside the 'cbo_budget_economic_outlook_mainpdfs' folder, strips away formatting

    layouts based on your notebook design, groups paragraphs using block bounds,
    and dynamically stitches cut-off lines/run-on sentences together.
    """
    # 1. Target the directory structure mirroring your notebook
    project_dir = Path.cwd()
    reports_dir = project_dir / "cbo_budget_economic_outlook_mainpdfs"
    pdf_path = reports_dir / pdf_filename

    # Validation check
    if not pdf_path.exists():
        raise FileNotFoundError(
            f"Could not find the file at: {pdf_path}\n"
            f"Please verify that the file is inside the 'cbo_budget_economic_outlook_mainpdfs' folder."
        )

    print(f"Successfully located and opening: {pdf_path}")

    # 2. Extract content page by page using original block structures
    doc = fitz.open(pdf_path)
    raw_chunks = []

    for page in doc:
        blocks = page.get_text("blocks")

        for b in blocks:
            text_block = b[4].strip()  # Grab the text content string

            # --- Inline Cleaning Filter (From your notebook) ---
            text_block = re.sub(
                r"(The Budget and Economic Outlook:|Economic Projections:|Chapter \d+)",
                "",
                text_block,
                flags=re.IGNORECASE,
            )
            text_block = re.sub(
                r"(---\s*PAGE\s*\\d+\s*---|\\b\d+\b\\s*$)", "", text_block
            )

            # Standardize cross-platform line breaks immediately
            text_block = text_block.replace("\r\n", "\n").replace("\r", "\n")

            if text_block.strip():
                raw_chunks.append(text_block.strip())

    # 3. Smart Stitching Pipeline to address cutoffs and structural fragment breaks
    stitched_paragraphs = []
    buffer_text = ""

    for chunk in raw_chunks:
        # Normalize internal line breaks inside the chunk to continuous spaces
        cleaned_chunk = re.sub(r"\s+", " ", chunk).strip()

        if not cleaned_chunk:
            continue

        if buffer_text:
            # If the ongoing buffer doesn't end with a terminal sentence boundary,
            # or if it ends with a hyphen, stitch the current chunk straight onto it.
            last_char = buffer_text[-1]
            is_cutoff = last_char not in [".", "!", "?", '"', "”"] or buffer_text.endswith("-")

            if is_cutoff:
                if buffer_text.endswith("-"):
                    buffer_text = buffer_text[:-1] + cleaned_chunk
                else:
                    buffer_text += " " + cleaned_chunk
                continue
            else:
                # Buffer was a fully closed paragraph sentence, save it
                stitched_paragraphs.append(buffer_text)

        buffer_text = cleaned_chunk

    # Catch the remaining trailing text block
    if buffer_text:
        stitched_paragraphs.append(buffer_text)

    # 4. Format into Requested Output Schema
    paragraph_rows = []
    global_paragraph_counter = 1

    for p_text in stitched_paragraphs:
        final_clean = re.sub(r"\s+", " ", p_text).strip()

        if final_clean:
            paragraph_rows.append(
                {
                    "report_name": pdf_filename,
                    "paragraph_number": global_paragraph_counter,
                    "text": final_clean,
                }
            )
            global_paragraph_counter += 1

    return pd.DataFrame(paragraph_rows)

In [3]:
cleaned_report_2026 = process_pdf_by_paragraph("2026-02-11__61882__The Budget and Economic Outlook_ 2026 to 2036.pdf")

Successfully located and opening: C:\Users\ericsc\Documents\GitHub\projection-uncertainty\cbo_budget_economic_outlook_mainpdfs\2026-02-11__61882__The Budget and Economic Outlook_ 2026 to 2036.pdf


### Iterate through 2000-2026 pdfs to group by paragraphs and categorize them using word2vec and cosine similiarity

In [4]:
import os
import re
from pathlib import Path
import fitz
import numpy as np
import pandas as pd

# ======================================================================
# 1. Smarter Paragraph Stitching Engine
# ======================================================================


def process_pdf_by_paragraph(pdf_path, pdf_filename):
    """Locates a PDF inside the 'cbo_budget_economic_outlook_mainpdfs' folder, strips away formatting
    layouts based on your notebook design, groups paragraphs using block bounds,
    and dynamically stitches cut-off lines/run-on sentences together.
    """
    try:
        doc = fitz.open(pdf_path)
    except Exception as e:
        print(f"❌ Failed to open ")
        return []

    raw_chunks = []

    # 1. Extract content page by page using original block structures
    for page in doc:
        blocks = page.get_text("blocks")

        for b in blocks:
            text_block = b[4].strip()  # Grab the text content string

            # --- Inline Cleaning Filter (From your notebook) ---
            text_block = re.sub(
                r"(The Budget and Economic Outlook:|Economic Projections:|Chapter \d+)",
                "",
                text_block,
                flags=re.IGNORECASE,
            )
            text_block = re.sub(
                r"(---\s*PAGE\s*\d+\s*---|\\b\d+\b\\s*$)", "", text_block
            )

            # Standardize cross-platform line breaks immediately
            text_block = text_block.replace("\r\n", "\n").replace("\r", "\n")

            if text_block.strip():
                raw_chunks.append(text_block.strip())

    doc.close()

    # 2. Smart Stitching Pipeline to address cutoffs and structural fragment breaks
    stitched_paragraphs = []
    buffer_text = ""

    for chunk in raw_chunks:
        # Normalize internal line breaks inside the chunk to continuous spaces
        cleaned_chunk = re.sub(r"\s+", " ", chunk).strip()

        if not cleaned_chunk:
            continue

        if buffer_text:
            # If the ongoing buffer doesn't end with a terminal sentence boundary,
            # or if it ends with a hyphen, stitch the current chunk straight onto it.
            last_char = buffer_text[-1]
            is_cutoff = last_char not in [
                ".",
                "!",
                "?",
                '"',
                "”",
            ] or buffer_text.endswith("-")

            if is_cutoff:
                if buffer_text.endswith("-"):
                    buffer_text = buffer_text[:-1] + cleaned_chunk
                else:
                    buffer_text += " " + cleaned_chunk
                continue
            else:
                # Buffer was a fully closed paragraph sentence, save it
                stitched_paragraphs.append(buffer_text)

        buffer_text = cleaned_chunk

    # Catch the remaining trailing text block
    if buffer_text:
        stitched_paragraphs.append(buffer_text)

    # 3. Format into Requested Output Schema (Paragraph counter resets for each report)
    paragraph_rows = []
    paragraph_counter = 1

    for p_text in stitched_paragraphs:
        final_clean = re.sub(r"\s+", " ", p_text).strip()

        if final_clean:
            paragraph_rows.append(
                {
                    "report_name": pdf_filename,
                    "paragraph_number": paragraph_counter,
                    "text": final_clean,
                    "component": pd.NA,
                    "category": pd.NA,
                    "subcategory": pd.NA,
                }
            )
            paragraph_counter += 1

    return paragraph_rows


# ======================================================================
# 2. Batch Execution Pipeline (2000-2026 Verification)
# ======================================================================

project_dir = Path.cwd()
reports_dir = project_dir / "cbo_budget_economic_outlook_mainpdfs"

if not reports_dir.exists() or not reports_dir.is_dir():
    raise ValueError(
        f"The directory '{reports_dir}' does not exist. Please check your path structure."
    )

# Regex ignores strict boundaries so numbers touching characters match seamlessly
year_pattern = re.compile(r"(20[0-2][0-9])")

# Build case-insensitive list of PDF files
pdf_files = [
    p for p in reports_dir.iterdir() if p.is_file() and p.suffix.lower() == ".pdf"
]
pdf_files.sort()

all_pdf_rows = []
print(
    f"Found {len(pdf_files)} report(s) inside '{reports_dir.name}'. Filtering for 2000-2026...\n"
    + "-" * 60
)

for file_path in pdf_files:
    pdf_filename = file_path.name
    match = year_pattern.search(pdf_filename)

    if match:
        try:
            report_rows = process_pdf_by_paragraph(file_path, pdf_filename)
            all_pdf_rows.extend(report_rows)
            print(f"✅ Successfully processed paragraphs: {pdf_filename}")
        except Exception as e:
            print(f"❌ Error extracting data from {pdf_filename}: {e}")
    else:
        print(f"⏭️ Skipping (Outside 2000-2026 window): {pdf_filename}")

# Generate DataFrame
df2 = pd.DataFrame(all_pdf_rows)

# Specify your directory path
dir_path = Path("data_files")
dir_path.mkdir(parents=True, exist_ok=True)
df2.to_csv('data_files/chunked_paragraphs.csv', index=False)

Found 71 report(s) inside 'cbo_budget_economic_outlook_mainpdfs'. Filtering for 2000-2026...
------------------------------------------------------------
✅ Successfully processed paragraphs: 2000-01-01__12069__The Budget and Economic Outlook_ Fiscal Years 2001-2010 (Table 1-6 corrected 2_1_00).pdf
✅ Successfully processed paragraphs: 2000-07-01__12477__The Budget and Economic Outlook_ An Update.pdf
✅ Successfully processed paragraphs: 2001-01-01__12958__The Budget and Economic Outlook_ Fiscal Years 2002-2011.pdf
✅ Successfully processed paragraphs: 2001-08-01__13249__The Budget and Economic Outlook_ An Update.pdf
✅ Successfully processed paragraphs: 2002-01-01__13504__The Budget and Economic Outlook_ Fiscal Years 2003-2012.pdf
✅ Successfully processed paragraphs: 2002-08-27__13956__The Budget and Economic Outlook_ An Update.pdf
✅ Successfully processed paragraphs: 2003-01-01__14254__The Budget and Economic Outlook_ Fiscal Years 2004-2013.pdf
✅ Successfully processed paragraphs: 2003-08